<a href="https://colab.research.google.com/github/joshstructure/D2MPDB/blob/main/Fence_Wind_Pier_Loads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C002 - Fence Wind and Pier Node Loads

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshstructure/D2MPDB/blob/main/Fence_Wind_Pier_Loads.ipynb)

Portable Colab edition for the **D2 I-95 pedestrian bridge**: the journal calculation C002 as a notebook. Choose **Runtime → Run all**. The calculation package is embedded in this file.

**What it does.** The wind loads transferred to the intermediate pier: the prescribed LRFD 13.8.2 pressure on the gross projected fence face, the concrete superstructure wind from C001 over the pier tributary length, the simplified LRFD 3.8.1.2.3a components (100 % transverse plus 25 % longitudinal, applied simultaneously) at the cap reference for Strength III and Service I, and the separate wind acting directly on the pier wall and cap.

**Workflow.** Cell 1 shows the inputs with their journal module or code provision and loads a saved case. Cell 2 holds the inputs, results, the journal figures, the D/C register with *Equation & values* disclosures, the equations and the exports (case JSON, calculation report, checks CSV, distilled Blockpad .bpad with its verification record, full bundle).

In [ ]:
# @title Disable output scrolling
from IPython.display import Javascript, display
from IPython import get_ipython

def disable_colab_scrolling(*_):
    display(Javascript('google.colab.output.setIframeHeight(0, true, {maxHeight: 10000})'))

_scroll_events = get_ipython().events
for _callback in list(_scroll_events.callbacks.get('pre_run_cell', [])):
    if getattr(_callback, '__name__', '') == 'disable_colab_scrolling':
        _scroll_events.unregister('pre_run_cell', _callback)
_scroll_events.register('pre_run_cell', disable_colab_scrolling)
print("Output scrolling disabled for future cell executions.")

In [ ]:
# @title Start the portable tool
from pathlib import Path
import base64, hashlib, importlib, io, os, subprocess, sys, zipfile
try:
    from google.colab import output as colab_output
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
revision = "1c7779b021434c92a76a084f58d1c43ec7353b085badd0d3d03aac6188640502"
PACKAGE = "..."
_package_bytes = base64.b64decode(PACKAGE)
if hashlib.sha256(_package_bytes).hexdigest() != revision:
    raise RuntimeError("Embedded calculation package is damaged.")
ROOT = (Path("/content") if IN_COLAB else Path.cwd()) / ("fence-tool-" + revision[:12])
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_package_bytes)) as _package:
    for _name in _package.namelist():
        _target = (ROOT / _name).resolve()
        if not _target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Invalid embedded package path.")
        _contents = _package.read(_name)
        if _target.exists() and _target.read_bytes() != _contents:
            raise RuntimeError("An extracted calculation file was edited. Preserve your edits and exports before using a fresh notebook folder.")
        _target.parent.mkdir(parents=True, exist_ok=True)
        if not _target.exists():
            _target.write_bytes(_contents)
if IN_COLAB:
    requirements = ROOT / "requirements-pier-cap-colab.txt"
    if not (ROOT / "fence_wind").is_dir() or not requirements.is_file():
        raise RuntimeError("The embedded package is incomplete. Reopen the delivered notebook.")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(requirements)], check=True)
    import ipywidgets as widget_runtime
    from importlib.metadata import version as installed_version
    loaded_widgets = widget_runtime.__version__
    installed_widgets = installed_version("ipywidgets")
    if loaded_widgets != installed_widgets or loaded_widgets != "7.7.1":
        raise RuntimeError(
            f"Widget session restart required: Python loaded ipywidgets {loaded_widgets}, "
            f"but {installed_widgets} is installed. Use Runtime → Restart session, "
            "then reload this browser page and Run all. Save any existing case before restarting."
        )
    import plotly as plot_runtime
    if plot_runtime.__version__ != "5.24.1" or installed_version("plotly") != "5.24.1":
        raise RuntimeError(
            "Chart session restart required. Save your case, use Runtime → Restart session, "
            "reload this browser page, then Run all. Colab needs Plotly 5.24.1 for live chart widgets."
        )
    widget_runtime.register_comm_target(get_ipython().kernel)
    colab_output.enable_custom_widget_manager()
    os.chdir(ROOT)
    for module_name in list(sys.modules):
        if module_name.split(".")[0] in ("pier_cap", "calc_common", "fence_wind", "scripts", "report_cards"):
            del sys.modules[module_name]
    importlib.invalidate_caches()
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))
from fence_wind.widgets import Notebook
from fence_wind.model import evaluate, default_case
print(f"Ready · Fence Wind and Pier Node Loads package {revision[:12]}. Run Cell 1 and Cell 2.")

In [ ]:
# @title Cell 1 · Project sources and assumptions
from copy import deepcopy
# Keep the active case, confirmations and reviews if this cell is run again.
_previous_app = globals().get("app")
_case = deepcopy(_previous_app.case) if _previous_app is not None else None
if _previous_app is not None:
    _previous_app.close()
app = Notebook(_case, export_root=ROOT / "exports")
app.display("sources");

In [ ]:
# @title Cell 2 · Fence Wind and Pier Node Loads
if "app" not in globals():
    print("Run Cell 1 first to load the project sources, then run this cell.")
else:
    app.display("design");